In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/panther_features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.shape

C:\Users\HP\AppData\Local\Temp\ipykernel_4724\1046593695.py:6: DtypeWarning: Columns (0: water, 1: eui) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/panther_features.csv")


(1842120, 33)

In [3]:
required_cols = ['meter_reading', 'lag_1h', 'lag_24h', 'lag_168h', 
                   'rolling_24h_mean', 'rolling_168h_mean']

model_df = df.dropna(subset=required_cols)
model_df.shape

(1452292, 33)

In [4]:
model_df['timestamp'].min(), model_df['timestamp'].max()

(Timestamp('2016-01-08 00:00:00'), Timestamp('2017-12-31 23:00:00'))

In [5]:
split_date = '2017-10-01'

train = model_df[model_df['timestamp'] < split_date]
test = model_df[model_df['timestamp'] >= split_date]

train.shape, test.shape

((1222594, 33), (229698, 33))

In [6]:
feature_cols = [
    'hour', 'day_of_week', 'month', 'is_weekend',
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_24h_mean', 'rolling_168h_mean',
    'airTemperature', 'dewTemperature', 'windSpeed',
    'sqm', 'primaryspaceusage', 'building_id'
]

target_col = 'meter_reading'

X_train = train[feature_cols]
y_train = train[target_col]
X_test = test[feature_cols]
y_test = test[target_col]

X_train.dtypes

hour                   int64
day_of_week            int64
month                  int64
is_weekend             int64
lag_1h               float64
lag_24h              float64
lag_168h             float64
rolling_24h_mean     float64
rolling_168h_mean    float64
airTemperature       float64
dewTemperature       float64
windSpeed            float64
sqm                  float64
primaryspaceusage        str
building_id              str
dtype: object

In [7]:
categorical_cols = ['primaryspaceusage', 'building_id']

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

X_train.dtypes

hour                    int64
day_of_week             int64
month                   int64
is_weekend              int64
lag_1h                float64
lag_24h               float64
lag_168h              float64
rolling_24h_mean      float64
rolling_168h_mean     float64
airTemperature        float64
dewTemperature        float64
windSpeed             float64
sqm                   float64
primaryspaceusage    category
building_id          category
dtype: object

In [8]:
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train, y_train,
    categorical_feature=categorical_cols
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1706
[LightGBM] [Info] Number of data points in the train set: 1222594, number of used features: 15
[LightGBM] [Info] Start training from score 107.437111


,learning_rate,0.05
,n_estimators,200
,objective,'regression'
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [9]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    'timestamp': test['timestamp'].values,
    'building_id': test['building_id'].values,
    'actual': y_test.values,
    'predicted': y_pred
})

results.head(10)


,timestamp,building_id,actual,predicted
0,2017-10-01 00:00:00,Panther_assembly_Carrol,13.8027,52.410016
1,2017-10-01 01:00:00,Panther_assembly_Carrol,11.5022,12.967275
2,2017-10-01 02:00:00,Panther_assembly_Carrol,11.4022,12.060957
3,2017-10-01 03:00:00,Panther_assembly_Carrol,11.5022,12.767658
4,2017-10-01 04:00:00,Panther_assembly_Carrol,11.4022,12.835539
5,2017-10-01 05:00:00,Panther_assembly_Carrol,11.2022,12.507696
6,2017-10-01 06:00:00,Panther_assembly_Carrol,26.3051,18.259719
7,2017-10-01 07:00:00,Panther_assembly_Carrol,34.0066,32.866322
8,2017-10-01 08:00:00,Panther_assembly_Carrol,35.8069,35.456047
9,2017-10-01 09:00:00,Panther_assembly_Carrol,36.6071,36.455622


In [10]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")

MAPE: 0.1084
RMSE: 9.6635


In [11]:
quantiles = [0.1, 0.5, 0.9]
quantile_models = {}

for q in quantiles:
    print(f"Training quantile {q}...")
    qmodel = lgb.LGBMRegressor(
        objective='quantile',
        alpha=q,
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )
    qmodel.fit(X_train, y_train, categorical_feature=categorical_cols)
    quantile_models[q] = qmodel

print("Done training all quantile models.")

Training quantile 0.1...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026658 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1706
[LightGBM] [Info] Number of data points in the train set: 1222594, number of used features: 15
[LightGBM] [Info] Start training from score 9.363800
Training quantile 0.5...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1706
[LightGBM] [Info] Number of data points in the train set: 1222594, number of used features: 15
[LightGBM] [Info] Start training from score 64.212402
Training quantile 0.9...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026719 seconds.
You 

In [12]:
results_q = pd.DataFrame({
    'timestamp': test['timestamp'].values,
    'building_id': test['building_id'].values,
    'actual': y_test.values,
    'pred_lower': quantile_models[0.1].predict(X_test),
    'pred_median': quantile_models[0.5].predict(X_test),
    'pred_upper': quantile_models[0.9].predict(X_test)
})

results_q.head(10)

,timestamp,building_id,actual,pred_lower,pred_median,pred_upper
0,2017-10-01 00:00:00,Panther_assembly_Carrol,13.8027,13.193150,63.605526,75.759422
1,2017-10-01 01:00:00,Panther_assembly_Carrol,11.5022,9.712888,13.348705,18.824939
2,2017-10-01 02:00:00,Panther_assembly_Carrol,11.4022,9.740290,11.093106,18.824939
3,2017-10-01 03:00:00,Panther_assembly_Carrol,11.5022,9.736919,11.101188,18.824939
4,2017-10-01 04:00:00,Panther_assembly_Carrol,11.4022,10.510861,11.225582,18.824939
5,2017-10-01 05:00:00,Panther_assembly_Carrol,11.2022,9.784502,10.853842,18.824939
6,2017-10-01 06:00:00,Panther_assembly_Carrol,26.3051,11.175240,22.953111,28.336533
7,2017-10-01 07:00:00,Panther_assembly_Carrol,34.0066,24.278821,31.216301,37.306197
8,2017-10-01 08:00:00,Panther_assembly_Carrol,35.8069,31.603802,34.131319,39.156028
9,2017-10-01 09:00:00,Panther_assembly_Carrol,36.6071,33.276117,34.941905,39.470956


In [13]:
test_sample = test.sample(5000, random_state=42)
X_test_sample = test_sample[feature_cols]
for col in categorical_cols:
    X_test_sample[col] = X_test_sample[col].astype('category')

results_q_sample = pd.DataFrame({
    'timestamp': test_sample['timestamp'].values,
    'building_id': test_sample['building_id'].values,
    'actual': test_sample['meter_reading'].values,
    'pred_lower': quantile_models[0.1].predict(X_test_sample),
    'pred_median': quantile_models[0.5].predict(X_test_sample),
    'pred_upper': quantile_models[0.9].predict(X_test_sample)
})

results_q_sample.head(10)

,timestamp,building_id,actual,pred_lower,pred_median,pred_upper
0,2017-12-18 20:00:00,Panther_education_Enriqueta,144.9880,134.175046,146.706032,156.442131
1,2017-10-23 00:00:00,Panther_lodging_Fausto,39.9677,40.040486,42.636686,44.930862
2,2017-12-07 00:00:00,Panther_office_Valarie,38.4874,30.046794,33.991862,39.097068
3,2017-11-24 02:00:00,Panther_office_Lavinia,19.8438,19.280598,19.537904,22.006655
4,2017-12-20 23:00:00,Panther_office_Clementine,41.5630,38.522376,41.899888,48.451726
5,2017-12-09 00:00:00,Panther_lodging_Marisol,160.6810,148.852661,157.525274,174.537040
6,2017-10-18 22:00:00,Panther_parking_Jody,48.0893,47.274752,48.206687,49.738543
7,2017-12-05 01:00:00,Panther_office_Shauna,54.0104,53.330064,58.104450,62.982376
8,2017-10-20 16:00:00,Panther_education_Enriqueta,106.5806,105.773206,112.716141,121.386908
9,2017-10-26 04:00:00,Panther_parking_Stanley,54.6105,54.314382,54.905351,55.478532


In [14]:
y_pred_lower_full = quantile_models[0.1].predict(X_test)
y_pred_upper_full = quantile_models[0.9].predict(X_test)

coverage = ((y_test >= y_pred_lower_full) & (y_test <= y_pred_upper_full)).mean()
print(f"Coverage: {coverage:.4f}")

Coverage: 0.8299


In [15]:
import os
os.makedirs("../models", exist_ok=True)

model.booster_.save_model("../models/lgbm_median.txt")
for q in quantiles:
    quantile_models[q].booster_.save_model(f"../models/lgbm_quantile_{q}.txt")

print("All models saved.")

All models saved.


In [16]:
from prophet import Prophet

print("Prophet imported successfully")

Importing plotly failed. Interactive plots will not work.


Prophet imported successfully


In [17]:
building_name = 'Panther_education_Rosalie'

building_data = model_df[model_df['building_id'] == building_name][['timestamp', 'meter_reading']].copy()
building_data = building_data.rename(columns={'timestamp': 'ds', 'meter_reading': 'y'})
building_data = building_data.sort_values('ds')

building_data.shape
building_data.head()

,ds,y
459690,2016-05-27 18:00:00,52.4101
459691,2016-05-27 19:00:00,51.6099
459692,2016-05-27 20:00:00,51.8100
459693,2016-05-27 21:00:00,51.6099
459694,2016-05-27 22:00:00,51.2099


In [18]:
building_data.columns.tolist()
building_data.shape

(13934, 2)

In [19]:
split_date = '2017-10-01'

building_train = building_data[building_data['ds'] < split_date]
building_test = building_data[building_data['ds'] >= split_date]

print(building_train.shape, building_test.shape)

prophet_model = Prophet()
prophet_model.fit(building_train)

(11751, 2) (2183, 2)


09:40:20 - cmdstanpy - INFO - Chain [1] start processing
09:40:23 - cmdstanpy - INFO - Chain [1] done processing


In [20]:
future = building_test[['ds']].copy()
forecast = prophet_model.predict(future)

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head(10)

,ds,yhat,yhat_lower,yhat_upper
0,2017-10-01 00:00:00,56.908571,43.905473,69.235663
1,2017-10-01 01:00:00,57.062272,44.899367,70.324744
2,2017-10-01 02:00:00,57.078362,44.071799,69.826592
3,2017-10-01 03:00:00,56.871311,43.618847,69.085990
4,2017-10-01 04:00:00,56.606441,44.678274,70.010539
5,2017-10-01 05:00:00,56.564650,42.861456,69.793485
6,2017-10-01 06:00:00,56.853540,44.343759,70.106431
7,2017-10-01 07:00:00,57.249169,44.159137,68.565484
8,2017-10-01 08:00:00,57.346018,43.679111,70.178378
9,2017-10-01 09:00:00,56.900669,43.179359,69.704066


In [21]:
comparison = building_test[['ds', 'y']].merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
comparison.head(10)

,ds,y,yhat,yhat_lower,yhat_upper
0,2017-10-01 00:00:00,84.8164,56.908571,43.905473,69.235663
1,2017-10-01 01:00:00,84.4163,57.062272,44.899367,70.324744
2,2017-10-01 02:00:00,84.6163,57.078362,44.071799,69.826592
3,2017-10-01 03:00:00,86.4167,56.871311,43.618847,69.085990
4,2017-10-01 04:00:00,73.2141,56.606441,44.678274,70.010539
5,2017-10-01 05:00:00,83.0160,56.564650,42.861456,69.793485
6,2017-10-01 06:00:00,86.0166,56.853540,44.343759,70.106431
7,2017-10-01 07:00:00,85.4165,57.249169,44.159137,68.565484
8,2017-10-01 08:00:00,86.8167,57.346018,43.679111,70.178378
9,2017-10-01 09:00:00,86.0166,56.900669,43.179359,69.704066


In [22]:
prophet_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True
)
prophet_model.fit(building_train)

forecast = prophet_model.predict(future)

comparison = building_test[['ds', 'y']].merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
comparison.head(10)

09:40:25 - cmdstanpy - INFO - Chain [1] start processing
09:40:31 - cmdstanpy - INFO - Chain [1] done processing


,ds,y,yhat,yhat_lower,yhat_upper
0,2017-10-01 00:00:00,84.8164,71.605802,59.608885,84.188269
1,2017-10-01 01:00:00,84.4163,71.844430,60.960854,83.180101
2,2017-10-01 02:00:00,84.6163,71.938754,60.340128,83.487776
3,2017-10-01 03:00:00,86.4167,71.804420,59.802063,83.143233
4,2017-10-01 04:00:00,73.2141,71.611228,59.932978,82.735000
5,2017-10-01 05:00:00,83.0160,71.643476,59.748818,82.490987
6,2017-10-01 06:00:00,86.0166,72.008368,60.310369,84.489198
7,2017-10-01 07:00:00,85.4165,72.479029,61.293053,83.839375
8,2017-10-01 08:00:00,86.8167,72.648008,61.155203,84.464271
9,2017-10-01 09:00:00,86.0166,72.272965,61.258832,82.924675


In [23]:
building_train_clean = building_train[building_train['ds'] >= '2016-05-20']

prophet_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True
)
prophet_model.fit(building_train_clean)

forecast = prophet_model.predict(future)
comparison = building_test[['ds', 'y']].merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
comparison.head(10)

09:40:33 - cmdstanpy - INFO - Chain [1] start processing
09:40:39 - cmdstanpy - INFO - Chain [1] done processing


,ds,y,yhat,yhat_lower,yhat_upper
0,2017-10-01 00:00:00,84.8164,71.605802,60.893062,83.402339
1,2017-10-01 01:00:00,84.4163,71.844430,60.544921,83.250440
2,2017-10-01 02:00:00,84.6163,71.938754,59.795449,82.923600
3,2017-10-01 03:00:00,86.4167,71.804420,59.955994,83.838590
4,2017-10-01 04:00:00,73.2141,71.611228,59.946477,82.758025
5,2017-10-01 05:00:00,83.0160,71.643476,60.417639,84.837251
6,2017-10-01 06:00:00,86.0166,72.008368,59.693245,83.945738
7,2017-10-01 07:00:00,85.4165,72.479029,61.084511,84.410205
8,2017-10-01 08:00:00,86.8167,72.648008,62.469258,84.220749
9,2017-10-01 09:00:00,86.0166,72.272965,60.674436,83.096981


In [24]:
comparison['error'] = comparison['y'] - comparison['yhat']
comparison[['ds', 'y', 'yhat', 'error']].head(30)

,ds,y,yhat,error
0,2017-10-01 00:00:00,84.8164,71.605802,13.210598
1,2017-10-01 01:00:00,84.4163,71.844430,12.571870
2,2017-10-01 02:00:00,84.6163,71.938754,12.677546
3,2017-10-01 03:00:00,86.4167,71.804420,14.612280
4,2017-10-01 04:00:00,73.2141,71.611228,1.602872
5,2017-10-01 05:00:00,83.0160,71.643476,11.372524
6,2017-10-01 06:00:00,86.0166,72.008368,14.008232
7,2017-10-01 07:00:00,85.4165,72.479029,12.937471
8,2017-10-01 08:00:00,86.8167,72.648008,14.168692
9,2017-10-01 09:00:00,86.0166,72.272965,13.743635


In [25]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

prophet_mape = mean_absolute_percentage_error(comparison['y'], comparison['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(comparison['y'], comparison['yhat']))

print(f"Prophet MAPE: {prophet_mape:.4f}")
print(f"Prophet RMSE: {prophet_rmse:.4f}")

Prophet MAPE: 0.3676
Prophet RMSE: 26.9240


In [26]:
building_test_idx = test['building_id'] == building_name
lgbm_mape_single = mean_absolute_percentage_error(y_test[building_test_idx], y_pred[building_test_idx])
lgbm_rmse_single = np.sqrt(mean_squared_error(y_test[building_test_idx], y_pred[building_test_idx]))

print(f"LightGBM MAPE (this building): {lgbm_mape_single:.4f}")
print(f"LightGBM RMSE (this building): {lgbm_rmse_single:.4f}")

LightGBM MAPE (this building): 0.0479
LightGBM RMSE (this building): 4.1272


In [27]:
def evaluate_prophet_for_building(building_name, model_df, test, split_date='2017-10-01'):
    building_data = model_df[model_df['building_id'] == building_name][['timestamp', 'meter_reading']].copy()
    building_data = building_data.rename(columns={'timestamp': 'ds', 'meter_reading': 'y'})
    building_data = building_data.sort_values('ds')

    b_train = building_data[building_data['ds'] < split_date]
    b_test = building_data[building_data['ds'] >= split_date]

    pmodel = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
    pmodel.fit(b_train)

    future = b_test[['ds']].copy()
    forecast = pmodel.predict(future)

    comp = b_test[['ds', 'y']].merge(forecast[['ds', 'yhat']], on='ds')

    p_mape = mean_absolute_percentage_error(comp['y'], comp['yhat'])
    p_rmse = np.sqrt(mean_squared_error(comp['y'], comp['yhat']))

    idx = test['building_id'] == building_name
    l_mape = mean_absolute_percentage_error(y_test[idx], y_pred[idx])
    l_rmse = np.sqrt(mean_squared_error(y_test[idx], y_pred[idx]))

    print(f"--- {building_name} ---")
    print(f"Prophet  -> MAPE: {p_mape:.4f}, RMSE: {p_rmse:.4f}")
    print(f"LightGBM -> MAPE: {l_mape:.4f}, RMSE: {l_rmse:.4f}")
    
    return comp

In [28]:
df[df['primaryspaceusage'] == 'Office']['building_id'].unique()[:5]
df[df['primaryspaceusage'] == 'Lodging/residential']['building_id'].unique()[:5]

<ArrowStringArray>
[    'Panther_lodging_Alita', 'Panther_lodging_Anastasia',
    'Panther_lodging_Awilda',    'Panther_lodging_Blaine',
      'Panther_lodging_Cora']
Length: 5, dtype: str

In [29]:
print(df[df['primaryspaceusage'] == 'Office']['building_id'].unique()[:5])
print(df[df['primaryspaceusage'] == 'Lodging/residential']['building_id'].unique()[:5])

<ArrowStringArray>
['Panther_office_Antonette',     'Panther_office_Brent',
 'Panther_office_Catherine', 'Panther_office_Christian',
  'Panther_office_Christin']
Length: 5, dtype: str
<ArrowStringArray>
[    'Panther_lodging_Alita', 'Panther_lodging_Anastasia',
    'Panther_lodging_Awilda',    'Panther_lodging_Blaine',
      'Panther_lodging_Cora']
Length: 5, dtype: str


In [30]:
comp_office = evaluate_prophet_for_building('Panther_office_Brent', model_df, test)

09:40:42 - cmdstanpy - INFO - Chain [1] start processing
09:40:43 - cmdstanpy - INFO - Chain [1] done processing


--- Panther_office_Brent ---
Prophet  -> MAPE: 1.0839, RMSE: 8.0378
LightGBM -> MAPE: 0.3608, RMSE: 5.1007


In [31]:
comp_lodging = evaluate_prophet_for_building('Panther_lodging_Cora', model_df, test)

09:40:46 - cmdstanpy - INFO - Chain [1] start processing
09:40:49 - cmdstanpy - INFO - Chain [1] done processing


--- Panther_lodging_Cora ---
Prophet  -> MAPE: 0.0810, RMSE: 15.7378
LightGBM -> MAPE: 0.0339, RMSE: 6.0383


In [32]:
test_results = test[['building_id', 'primaryspaceusage']].copy()
test_results['actual'] = y_test.values
test_results['predicted'] = y_pred

def calc_building_metrics(group):
    mape = mean_absolute_percentage_error(group['actual'], group['predicted'])
    rmse = np.sqrt(mean_squared_error(group['actual'], group['predicted']))
    return pd.Series({'mape': mape, 'rmse': rmse})

per_building_metrics = test_results.groupby('building_id').apply(calc_building_metrics)
per_building_metrics.describe()

,mape,rmse
count,105.000000,105.000000
mean,0.109467,6.886634
std,0.236141,6.793473
min,0.013895,0.373350
25%,0.040772,3.210093
50%,0.056077,5.245153
75%,0.092071,8.511354
max,2.139874,48.939631


In [33]:
worst_buildings = per_building_metrics.sort_values('mape', ascending=False).head(10)
worst_buildings

,mape,rmse
building_id,,
Panther_office_Christin,2.139874,1.033407
Panther_parking_Jody,0.921665,5.456931
Panther_other_Lula,0.750873,0.473100
Panther_retail_Gilbert,0.468957,0.373350
Panther_education_Tina,0.361502,2.012391
Panther_office_Brent,0.360850,5.100714
Panther_office_Hannah,0.209261,1.360635
Panther_education_Jonathan,0.181335,0.760478
Panther_assembly_Carrol,0.178849,6.281782


In [34]:
df[df['building_id'] == 'Panther_office_Christin']['meter_reading'].describe()

count    13580.000000
mean        18.206162
std         21.075295
min          0.342100
25%          0.500100
50%         10.860100
75%         32.081950
max         87.628200
Name: meter_reading, dtype: float64

In [35]:
summary = pd.DataFrame({
    'Model': ['LightGBM (global, all buildings)', 'Prophet (Rosalie - Education)', 
              'Prophet (Brent - Office)', 'Prophet (Cora - Lodging)'],
    'MAPE': [per_building_metrics['mape'].median(), 0.3676, 1.0839, 0.0810],
    'RMSE': [per_building_metrics['rmse'].median(), 26.9240, 8.0378, 15.7378]
})
summary

,Model,MAPE,RMSE
0,"LightGBM (global, all buildings)",0.056077,5.245153
1,Prophet (Rosalie - Education),0.367600,26.924000
2,Prophet (Brent - Office),1.083900,8.037800
3,Prophet (Cora - Lodging),0.081000,15.737800


In [36]:
dashboard_data = test[['timestamp', 'building_id', 'primaryspaceusage', 'sqm', 
                          'meter_reading', 'airTemperature'] + feature_cols].copy()
dashboard_data['predicted'] = y_pred
dashboard_data['pred_lower'] = quantile_models[0.1].predict(X_test)
dashboard_data['pred_upper'] = quantile_models[0.9].predict(X_test)

dashboard_data.to_csv("../data/processed/dashboard_data.csv", index=False)
dashboard_data.shape

(229698, 24)